# 02 - Classical Allocations

**Author:** Sacha Huberty

**Purpose:** Build the full S1-S5 classical allocation toolkit (efficient
frontier, Max Sharpe/CML, utility curves, GMV, Risk Parity, HRP,
tracking-error tracker, Permanent Portfolio and 60/40 benchmarks) on
the screened universe, then wire it into the weekly backtester and run
every book out-of-sample. This is the project's baseline: per
PROJECT_STRUCTURE.md's build order, "after step 2 you always have a
working, measurable strategy."

**Last updated:** 2026-07-25

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

pd.options.display.float_format = '{:.4f}'.format

from atlas import allocation, backtest, data, metrics, universe

cfg = data.load_config()
cfg["general"], cfg["optimization"]

## Data

In [ ]:
# Universe screened in notebook 01, as of the in-sample boundary.
as_of_universe = pd.Timestamp(cfg["general"]["is_end_date"])
universe_df = universe.load_universe(as_of_universe)
tickers = universe_df.index.tolist()
class_bucket = universe_df["class_bucket"]

prices = data.download_prices(tickers, start=cfg["general"]["start_date"])
prices = data.align_calendars(prices)
returns = data.daily_returns(prices)
returns.tail()

In [ ]:
is_end = pd.Timestamp(cfg["general"]["is_end_date"])
oos_start = pd.Timestamp(cfg["general"]["oos_start_date"])
lookback = cfg["optimization"]["lookback_days"]
cov_method = cfg["optimization"]["covariance"]
risk_aversions = cfg["optimization"]["risk_aversion_levels"]

## Analysis / signal logic

### Static snapshot (in-sample diagnostic only)

Mean/covariance estimated on the trailing `lookback_days` window ending
at the in-sample boundary. This section is a toolkit demonstration, not
a tuning decision -- it is not used anywhere in the OOS backtest below,
and is explicitly an in-sample diagnostic per the project's performance-claims
rule.

In [ ]:
snapshot_window = returns.loc[:is_end].tail(lookback)
mu = allocation.mean_returns(snapshot_window)
cov = allocation.covariance_matrix(snapshot_window, method=cov_method)
mu.sort_values(ascending=False)

In [ ]:
frontier = allocation.efficient_frontier(mu, cov, cfg, n_points=30)
w_ms = allocation.max_sharpe(mu, cov, cfg)
ms_ret = float(w_ms @ mu)
ms_vol = metrics.portfolio_vol(w_ms, cov)

plt.figure(figsize=(8, 6))
plt.plot(frontier["vol"], frontier["ret"], label="Efficient frontier")
plt.scatter([ms_vol], [ms_ret], color="red", marker="*", s=200, label="Max Sharpe")
cml_x = np.linspace(0, frontier["vol"].max() * 1.2, 10)
slope = ms_ret / ms_vol
plt.plot(cml_x, slope * cml_x, linestyle="--", label="Capital Market Line")
plt.xlabel("Volatility (annualized)")
plt.ylabel("Return (annualized)")
plt.title("Efficient Frontier, Max Sharpe, and CML (in-sample diagnostic)")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
for aversion in risk_aversions:
    utility = frontier["ret"] - 0.5 * aversion * frontier["vol"] ** 2
    plt.plot(frontier["vol"], utility, label=f"A={aversion}")
plt.xlabel("Volatility (annualized)")
plt.ylabel("Utility U = E[r] - 0.5*A*sigma^2")
plt.title("Utility along the efficient frontier (S1/S3)")
plt.legend()
plt.show()

In [ ]:
w_gmv = allocation.gmv(cov, cfg)
w_rp = allocation.risk_parity(cov, cfg)
w_hrp = allocation.hrp(snapshot_window, cfg)
w_permanent = allocation.permanent(class_bucket)
w_6040 = allocation.sixty_forty(class_bucket)
w_te = allocation.tracking_error_min(cov, w_permanent, cfg)

books = pd.DataFrame({
    "max_sharpe": w_ms,
    "gmv": w_gmv,
    "risk_parity": w_rp,
    "hrp": w_hrp,
    "tracking_error_min": w_te,
    "permanent": w_permanent,
    "sixty_forty": w_6040,
})
books

In [ ]:
risk_contrib = w_rp.to_numpy() * (cov.to_numpy() @ w_rp.to_numpy())
pd.Series(risk_contrib, index=w_rp.index).sort_values().plot(
    kind="barh", figsize=(6, 6), title="Risk Parity: risk contributions"
)
plt.xlabel("Risk contribution")
plt.show()

In [ ]:
# Correlation-distance dendrogram underlying the HRP clustering above
# (display-only recompute of the same transform used inside hrp()).
corr = snapshot_window.corr()
dist = np.sqrt(0.5 * (1.0 - corr))
link = linkage(squareform(dist.values, checks=False), method="single")

plt.figure(figsize=(11, 5))
dendrogram(link, labels=corr.columns.tolist())
plt.title("HRP correlation-distance dendrogram (S4)")
plt.tight_layout()
plt.show()

In [ ]:
def snapshot_stats(w):
    aligned = w.reindex(mu.index).fillna(0.0)
    ret = float(aligned @ mu)
    vol = metrics.portfolio_vol(aligned, cov)
    return pd.Series({
        "exp_return": ret,
        "exp_vol": vol,
        "exp_sharpe": ret / vol if vol > 0 else np.nan,
    })

static_table = books.apply(snapshot_stats, axis=0).T
for aversion in risk_aversions:
    col = f"utility_A{aversion}"
    static_table[col] = (
        static_table["exp_return"] - 0.5 * aversion * static_table["exp_vol"] ** 2
    )
static_table.sort_values("exp_sharpe", ascending=False)

### Out-of-sample backtest: the benchmark baseline

Each book below is recomputed every Friday on its own trailing
`lookback_days` window (the "signal clock" -- no model persistence
needed for these closed-form/optimization books, so no walk-forward
fold machinery is required yet; that lands in stage 9). The backtest
runs from `lookback_days` before the OOS boundary (so the first OOS
decision already has a full window) through the most recent data,
but only OOS-dated returns are used for every metric and chart below.

In [ ]:
def make_classical_strategy(method):
    def strategy_fn(as_of, window):
        w = window.tail(lookback)
        mu_t = allocation.mean_returns(w)
        cov_t = allocation.covariance_matrix(w, method=cov_method)
        if method == "max_sharpe":
            return allocation.max_sharpe(mu_t, cov_t, cfg)
        if method == "gmv":
            return allocation.gmv(cov_t, cfg)
        if method == "risk_parity":
            return allocation.risk_parity(cov_t, cfg)
        if method == "hrp":
            return allocation.hrp(w, cfg)
        raise ValueError(method)
    return strategy_fn


def permanent_strategy(as_of, window):
    return allocation.permanent(class_bucket)


def sixty_forty_strategy(as_of, window):
    return allocation.sixty_forty(class_bucket)


strategy_fns = {
    "max_sharpe": make_classical_strategy("max_sharpe"),
    "gmv": make_classical_strategy("gmv"),
    "risk_parity": make_classical_strategy("risk_parity"),
    "hrp": make_classical_strategy("hrp"),
    "permanent": permanent_strategy,
    "sixty_forty": sixty_forty_strategy,
}

In [ ]:
buffer_start_pos = max(0, returns.index.searchsorted(oos_start) - lookback)
backtest_returns = returns.iloc[buffer_start_pos:]

full_results = {
    name: backtest.run(fn, backtest_returns, cfg)
    for name, fn in strategy_fns.items()
}

## Results

In [ ]:
def oos_metrics(result):
    r = result.daily_returns.loc[oos_start:]
    return {
        "ann_return": metrics.ann_return(r),
        "ann_vol": metrics.ann_vol(r),
        "sharpe": metrics.sharpe(r),
        "sortino": metrics.sortino(r),
        "calmar": metrics.calmar(r),
        "max_drawdown": metrics.max_drawdown(r),
        "hit_rate": metrics.hit_rate(r),
        "avg_weekly_turnover": result.turnover.loc[oos_start:].mean(),
        "total_cost_drag": result.costs.loc[oos_start:].sum(),
    }


comparison = pd.DataFrame(
    {name: oos_metrics(res) for name, res in full_results.items()}
).T
comparison.sort_values("sharpe", ascending=False)

In [ ]:
plt.figure(figsize=(11, 6))
for name, res in full_results.items():
    oos_curve = (1.0 + res.daily_returns.loc[oos_start:]).cumprod()
    oos_curve.plot(label=name)
plt.title("OOS equity curves, rebased to 1.0 at the OOS start (baseline)")
plt.ylabel("Growth of $1")
plt.legend()
plt.show()

In [ ]:
full_results["risk_parity"].weights.loc[oos_start:].plot.area(
    figsize=(11, 5), title="Risk Parity: OOS weight evolution"
)
plt.ylabel("Weight")
plt.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
plt.show()

## Notes / next steps

- This is the project's baseline per the build order: a working,
  measurable strategy exists from this stage on, and every later stage
  must prove its OOS improvement against it (the ablation study in
  notebook 11).
- The static snapshot section (efficient frontier, utility curves) is
  an in-sample diagnostic only -- it never feeds into the OOS backtest
  weights, which are recomputed independently every Friday from
  trailing data.
- `max_sharpe`/`gmv`/`risk_parity` re-fit is currently a plain
  recompute each week (no persisted model state), so no walk-forward
  fold machinery is needed yet; that lands in stage 9 once
  regime/anomaly models with real persistence enter the pipeline.
- Turnover is capped at 2%/week with a 0.5% no-trade band (S4); the
  very first rebalance funds the portfolio from cash and is exempt
  from the cap, matching how a real fund launch works.
- Next (stage 3): `regimes.py` (HMM primary + K-Means macro context),
  wired in as the first view (V1) with posture-based switching between
  these classical books.